###**Step 1:** New research notebook
###**Step 2:** Data Pipeline

In [1]:
from google.colab import drive
import os

# Mount your Drive
drive.mount('/content/drive')

# Path to the shortcut folder
folder_path = '/content/drive/MyDrive/FD004_Research/CMAPSSData'

# List files
files = os.listdir(folder_path)
print(files)


# Define Project Root
PROJECT_DIR = "/content/drive/MyDrive/FD004_Research"

# Create Folder Structure
os.makedirs(f"{PROJECT_DIR}/data", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/artifacts", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/models", exist_ok=True)

print(f"Project structure initialized at: {PROJECT_DIR}")

Mounted at /content/drive
['train_FD001.txt', 'test_FD001.txt', 'train_FD003.txt', 'test_FD003.txt', 'train_FD002.txt', 'test_FD002.txt', 'train_FD004.txt', 'test_FD004.txt', 'RUL_FD001.txt', 'RUL_FD002.txt', 'RUL_FD004.txt', 'RUL_FD003.txt', 'readme.txt', 'Damage Propagation Modeling.pdf']
Project structure initialized at: /content/drive/MyDrive/FD004_Research


#### **Objective:** Leakage-free, reproducible FD004 pipeline with engine-level split.
### **Section 1:**  Configuration & Reproducibility

In [2]:
import random
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# --- Seed & Determinism ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# --- Pipeline Hyperparameters ---
DATA_DIR = '/content/drive/MyDrive/CMAPSSData'
RUL_CLIP = 125
WINDOW_SIZE = 30
N_REGIMES = 6
VAL_SPLIT_RATIO = 0.20

# 14 standard informative sensors for C-MAPSS FD002/FD004 (dropping constant/low-variance sensors)
OPERATING_SETTINGS = ["setting_1", "setting_2", "setting_3"]
SELECTED_SENSORS = [
    "s_2", "s_3", "s_4", "s_7", "s_8", "s_9",
    "s_11", "s_12", "s_13", "s_14", "s_15", "s_17", "s_20", "s_21"
]
ALL_RAW_COLS = (
    ["unit_id", "time_cycles"]
    + OPERATING_SETTINGS
    + [f"s_{i}" for i in range(1, 22)]
)

### **Section 2:**  Load & Verify Raw Data

In [3]:
# Point directly to your mounted Google Drive directory
DATA_DIR = '/content/drive/MyDrive/CMAPSSData'

train_df_raw = pd.read_csv(
    os.path.join(DATA_DIR, "train_FD004.txt"),
    sep=r"\s+",
    header=None,
    names=ALL_RAW_COLS
)
test_df_raw = pd.read_csv(
    os.path.join(DATA_DIR, "test_FD004.txt"),
    sep=r"\s+",
    header=None,
    names=ALL_RAW_COLS
)
rul_test_raw = pd.read_csv(
    os.path.join(DATA_DIR, "RUL_FD004.txt"),
    sep=r"\s+",
    header=None,
    names=["true_rul"]
)

# --- Sanity Checks ---
print(f"Train Raw Shape : {train_df_raw.shape} | Engines: {train_df_raw['unit_id'].nunique()}")
print(f"Test Raw Shape  : {test_df_raw.shape}  | Engines: {test_df_raw['unit_id'].nunique()}")
print(f"RUL Ground Truth: {rul_test_raw.shape}  | Engines: {len(rul_test_raw)}")

assert not train_df_raw.isnull().values.any(), "Missing values found in train_FD004."
assert not test_df_raw.isnull().values.any(), "Missing values found in test_FD004."
assert len(rul_test_raw) == test_df_raw["unit_id"].nunique(), "RUL count mismatch with test engines."

Train Raw Shape : (61249, 26) | Engines: 249
Test Raw Shape  : (41214, 26)  | Engines: 248
RUL Ground Truth: (248, 1)  | Engines: 248


### **Section 3:** Compute Piecewise Linear RUL Targets

In [4]:
def add_piecewise_rul(df: pd.DataFrame, clip_val: int = 125) -> pd.DataFrame:
    """Computes RUL_{i,t} = T_i - t and applies upper saturation clipping."""
    df_out = df.copy()
    max_cycles = df_out.groupby("unit_id")["time_cycles"].transform("max")
    df_out["RUL_raw"] = max_cycles - df_out["time_cycles"]
    df_out["RUL"] = df_out["RUL_raw"].clip(upper=clip_val)
    return df_out

train_df_processed = add_piecewise_rul(train_df_raw, clip_val=RUL_CLIP)

# Ground truth RUL for test engines
test_max_cycles = test_df_raw.groupby("unit_id")["time_cycles"].max().reset_index()
test_max_cycles["true_rul"] = rul_test_raw["true_rul"]

###**Step 3:** Engine-level validation split

### **Section 4:** Engine-Level Train / Validation Split

In [5]:
unique_engines = train_df_processed["unit_id"].unique()
rng = np.random.RandomState(SEED)
shuffled_engines = rng.permutation(unique_engines)

n_val = int(len(shuffled_engines) * VAL_SPLIT_RATIO)
val_engine_ids = shuffled_engines[:n_val]
train_engine_ids = shuffled_engines[n_val:]

# CRITICAL LEAKAGE ASSERTION
assert set(train_engine_ids).isdisjoint(set(val_engine_ids)), "Data leakage: overlapping engine IDs!"

train_split_df = train_df_processed[train_df_processed["unit_id"].isin(train_engine_ids)].copy()
val_split_df = train_df_processed[train_df_processed["unit_id"].isin(val_engine_ids)].copy()
test_split_df = test_df_raw.copy()

# ==============================================================================
# Refined Disjointness & Isolation Verification
# ==============================================================================

# Explicitly namespace engines to reflect dataset origin
train_engine_namespaced = {f"train_FD004_unit_{uid}" for uid in train_engine_ids}
val_engine_namespaced   = {f"train_FD004_unit_{uid}" for uid in val_engine_ids}
test_engine_namespaced  = {f"test_FD004_unit_{uid}"  for uid in test_df_raw["unit_id"].unique()}

overlap_train_val = train_engine_namespaced.intersection(val_engine_namespaced)
overlap_train_test = train_engine_namespaced.intersection(test_engine_namespaced)

print(f"Train ∩ Validation Overlap : {len(overlap_train_val)} (Expected: 0)")
print(f"Train ∩ Test Overlap       : {len(overlap_train_test)} (Expected: 0)")

assert len(overlap_train_val) == 0, "FATAL: Train/Val leakage within train_FD004.txt!"
assert len(overlap_train_test) == 0, "FATAL: Namespaced namespace collision between train/test files!"

print(f"Engine Split -> Train: {len(train_engine_ids)} engines | Val: {len(val_engine_ids)} engines")

Train ∩ Validation Overlap : 0 (Expected: 0)
Train ∩ Test Overlap       : 0 (Expected: 0)
Engine Split -> Train: 200 engines | Val: 49 engines


### **Section 5:** Zero-Leakage Preprocessing & Regime Clustering

In [6]:
# 1. Fit Standard Scaler on Operating Settings (Train Only)
setting_scaler = StandardScaler()
train_settings_scaled = setting_scaler.fit_transform(train_split_df[OPERATING_SETTINGS])
val_settings_scaled = setting_scaler.transform(val_split_df[OPERATING_SETTINGS])
test_settings_scaled = setting_scaler.transform(test_split_df[OPERATING_SETTINGS])

# 2. Fit K-Means on Scaled Settings (Train Only)
kmeans = KMeans(n_clusters=N_REGIMES, random_state=SEED, n_init=10)
train_split_df["regime"] = kmeans.fit_predict(train_settings_scaled)
val_split_df["regime"] = kmeans.transform(val_settings_scaled).argmin(axis=1) # Closest centroid
test_split_df["regime"] = kmeans.transform(test_settings_scaled).argmin(axis=1)

# 3. Fit Sensor Normalization (Train Only)
sensor_scaler = StandardScaler()
train_split_df[SELECTED_SENSORS] = sensor_scaler.fit_transform(train_split_df[SELECTED_SENSORS])
val_split_df[SELECTED_SENSORS] = sensor_scaler.transform(val_split_df[SELECTED_SENSORS])
test_split_df[SELECTED_SENSORS] = sensor_scaler.transform(test_split_df[SELECTED_SENSORS])

FEATURE_COLS = SELECTED_SENSORS + ["regime"]
# ==============================================================================
# Preprocessing Isolation Assertions
# ==============================================================================

# Confirm sensor scaler was fitted only on train_split_df
assert hasattr(sensor_scaler, "mean_") and hasattr(sensor_scaler, "scale_"), "Scaler uninitialized!"
assert sensor_scaler.n_samples_seen_ == len(train_split_df), (
    f"Leakage check failed: Scaler saw {sensor_scaler.n_samples_seen_} rows, "
    f"but train_split_df has {len(train_split_df)} rows."
)

# Confirm setting scaler was fitted only on train_split_df
assert setting_scaler.n_samples_seen_ == len(train_split_df), (
    f"Leakage check failed: Setting scaler saw {setting_scaler.n_samples_seen_} rows, "
    f"but train_split_df has {len(train_split_df)} rows."
)

# Confirm KMeans was fitted exclusively on training settings
assert kmeans.cluster_centers_.shape == (N_REGIMES, len(OPERATING_SETTINGS))
print("All preprocessing isolation assertions passed.")


All preprocessing isolation assertions passed.


### **Section 6:** Rolling Window Generation (Window Size = 30)

In [7]:
def generate_windows_train_val(df: pd.DataFrame, window_size: int, feature_cols: list):
    """Extracts sliding windows strictly confined within individual engines."""
    X_list, y_list = [], []
    for engine_id, group in df.groupby("unit_id"):
        features = group[feature_cols].values
        ruls = group["RUL"].values
        n_samples = len(group)

        if n_samples < window_size:
            continue

        for t in range(window_size, n_samples + 1):
            X_list.append(features[t - window_size : t])
            y_list.append(ruls[t - 1])

    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)

def generate_windows_test(df: pd.DataFrame, rul_df: pd.DataFrame, window_size: int, feature_cols: list, clip_val: int):
    """
    Extracts the last window per test engine and pairs it with the true terminal RUL.
    If engine lifetime < window_size, pads backwards with earliest available cycle.
    """
    X_list, y_list = [], []
    for _, row in rul_df.iterrows():
        engine_id = int(row["unit_id"])
        true_rul = min(float(row["true_rul"]), clip_val)

        group = df[df["unit_id"] == engine_id]
        features = group[feature_cols].values
        n_samples = len(features)

        if n_samples >= window_size:
            window = features[-window_size:]
        else:
            pad_len = window_size - n_samples
            pad = np.repeat(features[0:1], pad_len, axis=0)
            window = np.vstack([pad, features])

        X_list.append(window)
        y_list.append(true_rul)

    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)

# Generate Window Tensors
X_train, y_train = generate_windows_train_val(train_split_df, WINDOW_SIZE, FEATURE_COLS)
X_val, y_val = generate_windows_train_val(val_split_df, WINDOW_SIZE, FEATURE_COLS)
X_test, y_test = generate_windows_test(test_split_df, test_max_cycles, WINDOW_SIZE, FEATURE_COLS, RUL_CLIP)

# ==============================================================================
# Regime Dynamics Inside 30-Cycle Windows
# ==============================================================================

# Extract regime index sequences: shape (N_windows, 30)
train_regime_windows = X_train[:, :, 14].astype(int)

# Count unique regimes per individual window
unique_regimes_per_window = np.array([
    len(np.unique(window)) for window in train_regime_windows
])

regime_switch_counts = pd.Series(unique_regimes_per_window).value_counts().sort_index()

print("Unique Regimes per 30-Cycle Window Distribution:")
for n_regs, count in regime_switch_counts.items():
    pct = (count / len(X_train)) * 100
    print(f"  {n_regs} distinct regime(s) in window: {count:6d} ({pct:5.2f}%)")

Unique Regimes per 30-Cycle Window Distribution:
  4 distinct regime(s) in window:      6 ( 0.01%)
  5 distinct regime(s) in window:   1435 ( 3.28%)
  6 distinct regime(s) in window:  42255 (96.70%)


### **Section 7:** Pipeline Export & Validation Summary


In [8]:
print("--- Final Dataset Dimensions ---")
print(f"X_train : {X_train.shape} | y_train : {y_train.shape}")
print(f"X_val   : {X_val.shape}   | y_val   : {y_val.shape}")
print(f"X_test  : {X_test.shape}  | y_test  : {y_test.shape}")

# Persist processed artifacts for subsequent baseline modeling
os.makedirs("./artifacts", exist_ok=True)
np.savez_compressed(
    "./artifacts/FD004_baseline_tensors.npz",
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    X_test=X_test, y_test=y_test
)
print("\nTensors saved to ./artifacts/FD004_baseline_tensors.npz")

--- Final Dataset Dimensions ---
X_train : (43696, 30, 15) | y_train : (43696,)
X_val   : (10332, 30, 15)   | y_val   : (10332,)
X_test  : (248, 30, 15)  | y_test  : (248,)

Tensors saved to ./artifacts/FD004_baseline_tensors.npz


In [9]:
print("Train engines:", len(train_engine_ids))
print("Val engines:", len(val_engine_ids))

assert set(train_engine_ids).isdisjoint(set(val_engine_ids))
print()

print(FEATURE_COLS)
print(len(FEATURE_COLS))
print()

print("Train NaN:", np.isnan(X_train).sum())
print("Val NaN:  ", np.isnan(X_val).sum())
print("Test NaN: ", np.isnan(X_test).sum())

print("Train Inf:", np.isinf(X_train).sum())
print("Val Inf:  ", np.isinf(X_val).sum())
print("Test Inf: ", np.isinf(X_test).sum())
print()

print(y_train.min(), y_train.max())
print(y_val.min(), y_val.max())
print(y_test.min(), y_test.max())

Train engines: 200
Val engines: 49

['s_2', 's_3', 's_4', 's_7', 's_8', 's_9', 's_11', 's_12', 's_13', 's_14', 's_15', 's_17', 's_20', 's_21', 'regime']
15

Train NaN: 0
Val NaN:   0
Test NaN:  0
Train Inf: 0
Val Inf:   0
Test Inf:  0

0.0 125.0
0.0 125.0
6.0 125.0


###**Step 4:** Final pipeline verification

### **Section 8:** Pipeline Verification & Dataset Summary

In [10]:
# Extract test engine IDs for complete summary verification
test_engine_ids = test_df_raw["unit_id"].unique()

print("========== DATASET SUMMARY ==========")
print(f"Number of train engines      : {len(train_engine_ids)}")
print(f"Number of validation engines : {len(val_engine_ids)}")
print(f"Number of test engines       : {len(test_engine_ids)}")

print(f"\nX_train shape : {X_train.shape} | y_train shape : {y_train.shape}")
print(f"X_val shape   : {X_val.shape}   | y_val shape   : {y_val.shape}")
print(f"X_test shape  : {X_test.shape}  | y_test shape  : {y_test.shape}")

print("\n========== ENGINE SPLIT CHECK ==========")
overlap_train_val = set(train_engine_ids).intersection(set(val_engine_ids))
overlap_train_test = set(train_engine_ids).intersection(set(test_engine_ids))

print(f"Train ∩ Validation engines : {overlap_train_val}")
print(f"Train ∩ Validation empty   : {len(overlap_train_val) == 0}")
print(f"Train ∩ Test empty         : {len(overlap_train_test) == 0}")

assert len(overlap_train_val) == 0, "ERROR: Train/Validation engine leakage detected!"

print("\n========== SENSOR NORMALIZATION CHECK (14 SELECTED SENSORS) ==========")
# Exclude the 15th feature column (regime integer index) from continuous stats
train_sensor_means = X_train[:, :, :14].mean(axis=(0, 1))
train_sensor_stds  = X_train[:, :, :14].std(axis=(0, 1))
val_sensor_means   = X_val[:, :, :14].mean(axis=(0, 1))
val_sensor_stds    = X_val[:, :, :14].std(axis=(0, 1))

norm_summary_df = pd.DataFrame({
    "Sensor": SELECTED_SENSORS,
    "Train Mean": np.round(train_sensor_means, 4),
    "Train Std":  np.round(train_sensor_stds, 4),
    "Val Mean":   np.round(val_sensor_means, 4),
    "Val Std":    np.round(val_sensor_stds, 4),
})
print(norm_summary_df.to_string(index=False))

print("\n========== REGIME DISTRIBUTION CHECK ==========")
# Check discrete regime index distribution across splits
train_regimes, train_counts = np.unique(X_train[:, :, 14], return_counts=True)
val_regimes, val_counts = np.unique(X_val[:, :, 14], return_counts=True)

regime_df = pd.DataFrame({
    "Regime": np.arange(N_REGIMES),
    "Train Count": [train_counts[list(train_regimes).index(r)] if r in train_regimes else 0 for r in range(N_REGIMES)],
    "Val Count":   [val_counts[list(val_regimes).index(r)] if r in val_regimes else 0 for r in range(N_REGIMES)]
})
print(regime_df.to_string(index=False))

========== DATASET SUMMARY ==========
Number of train engines      : 200
Number of validation engines : 49
Number of test engines       : 248

X_train shape : (43696, 30, 15) | y_train shape : (43696,)
X_val shape   : (10332, 30, 15)   | y_val shape   : (10332,)
X_test shape  : (248, 30, 15)  | y_test shape  : (248,)

========== ENGINE SPLIT CHECK ==========
Train ∩ Validation engines : set()
Train ∩ Validation empty   : True
Train ∩ Test empty         : False

========== SENSOR NORMALIZATION CHECK (14 SELECTED SENSORS) ==========
Sensor  Train Mean  Train Std  Val Mean  Val Std
   s_2     -0.0040     0.9984    0.0091   0.9983
   s_3     -0.0078     0.9990    0.0096   0.9984
   s_4     -0.0097     0.9983    0.0119   0.9993
   s_7     -0.0034     0.9992    0.0071   0.9962
   s_8     -0.0027     0.9980    0.0043   0.9997
   s_9     -0.0071     0.9991    0.0028   0.9985
  s_11     -0.0108     0.9988    0.0099   0.9991
  s_12     -0.0035     0.9993    0.0071   0.9962
  s_13     -0.0016    

In [11]:
import numpy as np
import os
from google.colab import drive

# Ensure Drive is mounted
drive.mount('/content/drive')

ARTIFACT_DIR = "/content/drive/MyDrive/FD004_Research/artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# Save the preprocessed tensors
np.savez_compressed(
    f"{ARTIFACT_DIR}/FD004_baseline_tensors.npz",
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    X_test=X_test,
    y_test=y_test
)

print(f"Artifact successfully saved to {ARTIFACT_DIR}/FD004_baseline_tensors.npz")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Artifact successfully saved to /content/drive/MyDrive/FD004_Research/artifacts/FD004_baseline_tensors.npz
